In [1]:
# Checking the GPU

import torch
print("PyTorch Version:",torch.__version__)
print("GPU Available:",torch.cuda.is_available())

if torch.cuda.is_available():
  device ="cuda"
  print("GPU Name:",torch.cuda.get_device_name(0))
else:
  device="GPU"
  print("No GPU Connected.Check your runtime settings")

print("Selected Device:",device)

PyTorch Version: 2.11.0+cu128
GPU Available: True
GPU Name: Tesla T4
Selected Device: cuda


In [ ]:
#Step 2: Installing the Libraries

%pip install -q transformers sentencepiece accelerate

In [2]:
import transformers
import sentencepiece
import accelerate

print("Transformers:",transformers.__version__)
print("SentencePiece:",sentencepiece.__version__)
print("Accelerate:",accelerate.__version__)

Transformers: 5.16.1
SentencePiece: 0.2.2
Accelerate: 1.14.0


In [3]:
# Step 3 : Load the Tokenizer and Inspect a Sentence

from transformers import AutoTokenizer

#The pretrained model which we'll use
model_name = "facebook/nllb-200-distilled-600M"

#loading its tokenizer and set the input language to English
tokenizer = AutoTokenizer.from_pretrained(model_name,src_lang="eng_Latn")

#Example Customer Support Message
text = "I Cannot log into my Account. Please help!"

#Convert the sentence inot token ids
inputs=tokenizer(text,return_tensors="pt")

#Convert IDs back to Token pieces.So,we can inspect them
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

print("Original Text:",text)
print("Tokens:",tokens)
print("\nToken IDs:",inputs["input_ids"])
print("\nAttention Mask:",inputs["attention_mask"])
print("\nInput Shape:",inputs["input_ids"].shape)

#Reconstruct the Text from IDs

decode_text = tokenizer.decode(inputs["input_ids"][0],
                               skip_special_tokens=True)
print("\nDecoded Text:",decode_text)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

Original Text: I Cannot log into my Account. Please help!
Tokens: ['eng_Latn', '▁I', '▁Cann', 'ot', '▁log', '▁into', '▁my', '▁Account', '.', '▁Please', '▁help', '!', '</s>']

Token IDs: tensor([[256047,    117, 164991,    124,  12040,   9174,   1537, 128640, 248075,
          62470,   8264, 248203,      2]])

Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Input Shape: torch.Size([1, 13])

Decoded Text: I Cannot log into my Account. Please help!


In [4]:
# Step 4: Loading the Pretrained Transformer

import torch
from transformers import AutoModelForSeq2SeqLM

#Loading the pretrained model using 16-bit floating point numbers
model =AutoModelForSeq2SeqLM.from_pretrained(model_name,dtype=torch.float16)

#Move the model to our GPU
model=model.to(device)

#set the model to evalaution mode for translation
model.eval()

#Check whether then model loaded correctly
print("Model Loaded Successfully!")
print("Model Class:",type(model).__name__)
print("Model Device:",model.device)
print("Model DataType:",model.dtype)
print("Encoder-Decoder Model:",model.config.is_encoder_decoder)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model Loaded Successfully!
Model Class: M2M100ForConditionalGeneration
Model Device: cuda:0
Model DataType: torch.float16
Encoder-Decoder Model: True


In [5]:
#Step 5: Generate your first  translation

#Setting the input message and its language
text = "I cannot log into my account. Please help!"
tokenizer.src_lang="eng_Latn"

#Converting the text into tensors and move them onto the GPU
inputs = tokenizer(text,return_tensors="pt").to(model.device)

#Get the token ID identifying Hindi as the Output language
target_language="hin_Deva"
target_language_id = tokenizer.convert_tokens_to_ids(target_language)

#Generate the translated Token IDs
with torch.inference_mode():
  translated_ids=model.generate(**inputs,forced_bos_token_id=target_language_id,max_new_tokens=100,do_sample=False)

#Convert the generated Ids into readable text
translation = tokenizer.decode(translated_ids[0],skip_special_tokens=True)

print("Original Text:",text)
print("Target Language:",target_language)
print("Generated Token IDs:",translated_ids)
print("Translation",translation)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original Text: I cannot log into my account. Please help!
Target Language: hin_Deva
Generated Token IDs: tensor([[     2, 256068,   7923,   6623, 119879,   1035,  50305, 248309,  15695,
           3600,    707,  12667, 248075,  56147,  27571, 248203,      2]],
       device='cuda:0')
Translation मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद!


In [6]:
# Check the exact source-language setting
print("Source language:", repr(tokenizer.src_lang))

# Inspect the tokens actually passed to the model
input_tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].tolist()
)

print("Input tokens:", input_tokens)

# Stop with a clear message if either check fails

assert tokenizer.src_lang == "eng_Latn", "Use exactly eng_Latn"
assert input_tokens[0] == "eng_Latn", "Recreate inputs after fixing src_lang"

print("Both source-language checks passed.")

Source language: 'eng_Latn'
Input tokens: ['eng_Latn', '▁I', '▁cannot', '▁log', '▁into', '▁my', '▁account', '.', '▁Please', '▁help', '!', '</s>']
Both source-language checks passed.


In [7]:
# Step 6: Compare Greedy Decoding and Beam Search

#Comparing One Candidate path with 4 candidate paths

for beam_count in [1,4]:

  with torch.inference_mode():
    output_ids=model.generate(
        **inputs,forced_bos_token_id=tokenizer.convert_tokens_to_ids("hin_Deva"),
        max_new_tokens=100,
        num_beams=beam_count,
        do_sample=False
    )

  translated_text = tokenizer.decode(output_ids[0],skip_special_tokens=True)


  print(f"\nNumber of beams:{beam_count}")
  print("Translation:",translated_text)


  #Check whether decoding hid any unknown tokens
  has_unknown = tokenizer.unk_token_id in output_ids[0].tolist()
  print("Contains unknown token:",has_unknown)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Number of beams:1
Translation: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद!
Contains unknown token: False

Number of beams:4
Translation: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद!
Contains unknown token: False


In [8]:
# 5.Inspect the Model's Embeddings

#Get the encoders's token embedding layer
embedding_layer = model.get_encoder().get_input_embeddings()

#Look up an embedding vector for each input token
with torch.inference_mode():
  token_embeddings = embedding_layer(inputs["input_ids"])

print("Token IDs Shape:",inputs["input_ids"].shape)
print("Embedding Table Shape:",embedding_layer.weight.shape)
print("Token Embeddings Shape:",token_embeddings.shape)

#Position 0 is the language marker;position "1" is the token for "I"
token_position=1
token_id = inputs["input_ids"][0,token_position].item()

print("\nToken:",tokenizer.convert_ids_to_tokens(token_id))
print("Token ID:",token_id)
print("First 10 Embedding Values:",token_embeddings[0,token_position,:10].float().cpu().tolist())

Token IDs Shape: torch.Size([1, 12])
Embedding Table Shape: torch.Size([256206, 1024])
Token Embeddings Shape: torch.Size([1, 12, 1024])

Token: ▁I
Token ID: 117
First 10 Embedding Values: [-0.6904296875, -0.375, 0.10430908203125, 0.00763702392578125, -0.0849609375, 1.216796875, 0.213623046875, 2.0234375, -5.453125, -0.125]


In [9]:
#Step 6: Inspect Positional Embeddings

#Get the encoder from our local model
encoder = model.get_encoder()

#Compute positional vectors for the current input
with torch.inference_mode():
  position_embeddings = encoder.embed_positions(inputs["input_ids"])

print("Position Layer:",type(encoder.embed_positions).__name__)
print("Token Embeddings Shape:",token_embeddings.shape)
print("Position Embeddings Shape:",position_embeddings.shape)

#Compare the Positional vectors at the first 2 positions
print("\nPosition 0 - first 10 values:",position_embeddings[0,0,:10].float().cpu().tolist())

print("\nPosition 1 - first 10 values:",position_embeddings[0,1,:10].float().cpu().tolist())

Position Layer: M2M100SinusoidalPositionalEmbedding
Token Embeddings Shape: torch.Size([1, 12, 1024])
Position Embeddings Shape: torch.Size([1, 12, 1024])

Position 0 - first 10 values: [0.9091796875, 0.92333984375, 0.9365234375, 0.94775390625, 0.9580078125, 0.96728515625, 0.97509765625, 0.9814453125, 0.9873046875, 0.99169921875]

Position 1 - first 10 values: [0.14111328125, 0.1939697265625, 0.2452392578125, 0.295166015625, 0.34326171875, 0.3896484375, 0.43408203125, 0.47705078125, 0.51806640625, 0.55712890625]


In [10]:
#ENcoder Self Attention
# Get the encoder by calling the method
encoder = model.get_encoder()

# Set the encoder to evaluation mode
encoder.eval()

with torch.inference_mode():

    # Look up the initial token embeddings
    initial_embeddings = encoder.get_input_embeddings()(
        inputs["input_ids"]
    )

    # Process the input through all encoder layers
    encoder_outputs = encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True
    )

# Retrieve the final encoder representations
contextual_embeddings = encoder_outputs.last_hidden_state

print("Initial embeddings shape:", initial_embeddings.shape)
print("Encoder output shape:", contextual_embeddings.shape)

# Select the token at position 1
token_position = 1
token_id = inputs["input_ids"][0, token_position].item()

print("\nSelected token:", tokenizer.convert_ids_to_tokens(token_id))

print(
    "\nInitial embedding — first 10 values:",
    initial_embeddings[0, token_position, :10].float().cpu().tolist()
)

print(
    "\nContextual representation — first 10 values:",
    contextual_embeddings[0, token_position, :10].float().cpu().tolist()
)

Initial embeddings shape: torch.Size([1, 12, 1024])
Encoder output shape: torch.Size([1, 12, 1024])

Selected token: ▁I

Initial embedding — first 10 values: [-0.6904296875, -0.375, 0.10430908203125, 0.00763702392578125, -0.0849609375, 1.216796875, 0.213623046875, 2.0234375, -5.453125, -0.125]

Contextual representation — first 10 values: [0.220947265625, 0.4111328125, 0.35986328125, 0.039337158203125, -0.12548828125, 0.140625, -0.10809326171875, 0.050140380859375, -0.344482421875, 0.017547607421875]


In [11]:
#Using an attention implementation that returns attention weights
model.set_attn_implementation("eager")

#Retrieve the encoder and set evaluation tone
encoder = model.get_encoder()
encoder.eval()

#Run the encoder and request the attention weights
with torch.inference_mode():
  encoder_outputs = encoder(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      return_dict=True,
      output_attentions=True
  )

#Select the first encoder layer's attention tensor
first_layer_attention = encoder_outputs.attentions[0]

print("Number of Encoder layers:",len(encoder_outputs.attentions))
print("Attention Tensor Shape:",first_layer_attention.shape)

#Retrieve readable tokens for the current input
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

#Inspect the first ehad and token at position at 1
head_index=0
query_position =1

attention_weights=first_layer_attention[
    0,head_index,query_position,:].float().cpu()

print("\nQuery Token:",tokens[query_position])
print("Head index:",head_index)
print("\nAttention to each input position:")

for token,weight in zip(tokens,attention_weights.tolist()):
  print(f"{token:15s}{weight:.4f}")

print("\nSum of Attention Weights:",attention_weights.sum().item())

Number of Encoder layers: 12
Attention Tensor Shape: torch.Size([1, 16, 12, 12])

Query Token: ▁I
Head index: 0

Attention to each input position:
eng_Latn       0.0139
▁I             0.0121
▁cannot        0.3777
▁log           0.4924
▁into          0.0301
▁my            0.0079
▁account       0.0018
.              0.0005
▁Please        0.0062
▁help          0.0389
!              0.0172
</s>           0.0012

Sum of Attention Weights: 0.9999442100524902


In [12]:
# Retrieve self-attention from the first encoder layer
attention = model.get_encoder().layers[0].self_attn

# Read the model's dimensions
model_dimension = model.config.d_model
number_of_heads = model.config.encoder_attention_heads
head_dimension = model_dimension // number_of_heads

print("Model dimension:", model_dimension)
print("Number of attention heads:", number_of_heads)
print("Dimension per head:", head_dimension)
print("Attention scaling divisor:", head_dimension ** 0.5)

# Inspect the learned query, key, and value projection matrices
print("\nQuery projection shape:", attention.q_proj.weight.shape)
print("Key projection shape:", attention.k_proj.weight.shape)
print("Value projection shape:", attention.v_proj.weight.shape)

# Inspect the projection applied after combining all heads
print("Output projection shape:", attention.out_proj.weight.shape)

Model dimension: 1024
Number of attention heads: 16
Dimension per head: 64
Attention scaling divisor: 8.0

Query projection shape: torch.Size([1024, 1024])
Key projection shape: torch.Size([1024, 1024])
Value projection shape: torch.Size([1024, 1024])
Output projection shape: torch.Size([1024, 1024])


In [13]:
#Residual Connection.layer normalisation,feed forward network

#Retrieve the first encoder layer
encoder=model.get_encoder()
layer=encoder.layers[0]

#Inspect the layer normaization modules
print("Normalization before self-attention:")
print(layer.self_attn_layer_norm)

print("\nNormalization after self-attention:")
print(layer.final_layer_norm)

#Inspect the Feed Forward network
print("\nFeed Forward network:")
print(layer.fc1)

print("\nActivation Function:")
print(model.config.activation_function)

print("\nFeed Forward network:")
print(layer.fc2)

#Show the feature dimensions clearly
print("\nFeed Forward dimensions:")
print(
    f"{layer.fc1.in_features}"
    f" → {layer.fc1.out_features}"
    f" → {layer.fc2.out_features}"
)

# Residual connections add vectors; they are not separate modules
print("\nResidual connections:")
print("H = X + SelfAttention(LayerNorm(X))")
print("Output = H + FeedForward(LayerNorm(H))")

Normalization before self-attention:
LayerNorm((1024,), eps=1e-05, elementwise_affine=True)

Normalization after self-attention:
LayerNorm((1024,), eps=1e-05, elementwise_affine=True)

Feed Forward network:
Linear(in_features=1024, out_features=4096, bias=True)

Activation Function:
relu

Feed Forward network:
Linear(in_features=4096, out_features=1024, bias=True)

Feed Forward dimensions:
1024 → 4096 → 1024

Residual connections:
H = X + SelfAttention(LayerNorm(X))
Output = H + FeedForward(LayerNorm(H))


In [17]:
# Decoder Part

#Use prediction behaviour
model.eval()

#Retrieve the model's decoder starting token
start_token_id = model.config.decoder_start_token_id

print("Decoder start token ID:",start_token_id)
print(
    "Decoderr Start token:",tokenizer.convert_ids_to_tokens(start_token_id)
)

#Create one drcoder sequence containgin only the start token
decoder_input_ids = torch.tensor(
    [[start_token_id]],
    dtype=torch.long,
    device = model.device
)

#Run one forward pass

with torch.inference_mode():
  outputs=model(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      decoder_input_ids=decoder_input_ids,
      output_hidden_states=True,
      return_dict=True
  )

print("\nDecoder input shape:",decoder_input_ids.shape)
print("Logits Shape:",outputs.logits.shape)

#Select vocabulary scores from the last decoder position
next_token_scores = outputs.logits[0,-1,:]

#Convert raw scores into probabilities
probabilities = torch.softmax(next_token_scores.float(),dim=-1)

#Retrieve the five higest probability next tokens
top_probabilities,top_ids = torch.topk(probabilities,k=5)

print("\nTop 5 raw next-token predictions:")

for token_id,probability in zip(
    top_ids.tolist(),
    top_probabilities.tolist()
):
  print(f"{tokenizer.convert_ids_to_tokens(token_id):15s}{probability:.4f}")

Decoder start token ID: 2
Decoderr Start token: </s>

Decoder input shape: torch.Size([1, 1])
Logits Shape: torch.Size([1, 1, 256206])

Top 5 raw next-token predictions:
arb_Arab       0.0972
pol_Latn       0.0550
ita_Latn       0.0481
bul_Cyrl       0.0445
nld_Latn       0.0431


In [18]:
# Set evaluation behaviour
model.eval()

# Retrieve the decoder start token and Hindi language token
start_token_id = model.config.decoder_start_token_id
hindi_token_id = tokenizer.convert_tokens_to_ids("hin_Deva")

# Create the two-token decoder prefix
decoder_input_ids = torch.tensor(
    [[start_token_id, hindi_token_id]],
    dtype=torch.long,
    device=model.device
)

print(
    "Decoder prefix:",
    tokenizer.convert_ids_to_tokens(decoder_input_ids[0].tolist())
)

# Predict using the English source and decoder prefix
with torch.inference_mode():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=decoder_input_ids,
        return_dict=True
    )

print("Logits shape:", outputs.logits.shape)

# Get probabilities for the token after the Hindi marker
next_token_scores = outputs.logits[0, -1, :]
probabilities = torch.softmax(next_token_scores.float(), dim=-1)

# Show the five highest-probability next tokens
top_probabilities, top_ids = torch.topk(probabilities, k=5)

print("\nTop 5 next-token candidates:")

for token_id, probability in zip(
    top_ids.tolist(),
    top_probabilities.tolist()
):
    token = tokenizer.convert_ids_to_tokens(token_id)
    print(f"{token:20s} ID: {token_id:<7d} Probability: {probability:.4f}")

# Select the highest-probability token
next_token_id = top_ids[0].item()

# Put it into a tensor with matching shape and device
next_token = torch.tensor(
    [[next_token_id]],
    dtype=torch.long,
    device=model.device
)

# Append it to the decoder prefix
extended_decoder_ids = torch.cat(
    [decoder_input_ids, next_token],
    dim=1
)

print(
    "\nSelected token:",
    tokenizer.convert_ids_to_tokens(next_token_id)
)

print(
    "Translation started:",
    tokenizer.decode(
        extended_decoder_ids[0],
        skip_special_tokens=True
    )
)

Decoder prefix: ['</s>', 'hin_Deva']
Logits shape: torch.Size([1, 2, 256206])

Top 5 next-token candidates:
▁मैं                 ID: 7923    Probability: 0.7042
▁मुझे                ID: 16859   Probability: 0.0436
▁मेरे                ID: 24602   Probability: 0.0319
▁मेरा                ID: 36605   Probability: 0.0143
▁मैंने               ID: 34544   Probability: 0.0099

Selected token: ▁मैं
Translation started: मैं


In [19]:
# Use evaluation behaviour
model.eval()
encoder = model.get_encoder()

# Set the target language
hindi_token_id = tokenizer.convert_tokens_to_ids("hin_Deva")

# Start the decoder with its start token and Hindi marker
decoder_ids = torch.tensor(
    [[model.config.decoder_start_token_id, hindi_token_id]],
    dtype=torch.long,
    device=model.device
)

max_steps = 50
reached_end = False

with torch.inference_mode():

    # Encode the source sentence only once
    source_outputs = encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True
    )

    # Generate one new token per iteration
    for step in range(max_steps):

        outputs = model(
            encoder_outputs=source_outputs,
            attention_mask=inputs["attention_mask"],
            decoder_input_ids=decoder_ids,
            use_cache=False,
            return_dict=True
        )

        # Select the highest-scoring next token
        next_token_id = outputs.logits[0, -1, :].argmax().item()

        # Create a tensor containing the selected token
        next_token = torch.tensor(
            [[next_token_id]],
            dtype=torch.long,
            device=model.device
        )

        # Append the token to the decoder sequence
        decoder_ids = torch.cat(
            [decoder_ids, next_token],
            dim=1
        )

        # Display the chosen token and the translation so far
        token_piece = tokenizer.convert_ids_to_tokens(next_token_id)

        current_text = tokenizer.decode(
            decoder_ids[0],
            skip_special_tokens=True
        )

        print(f"Step {step + 1}: {token_piece}")
        print("Text so far:", current_text)
        print()

        # Stop when the model generates the end marker
        if next_token_id == model.config.eos_token_id:
            reached_end = True
            break

final_translation = tokenizer.decode(
    decoder_ids[0],
    skip_special_tokens=True
)

print("Final translation:", final_translation)

if reached_end:
    print("Stopped because the model generated its end token.")
else:
    print("Reached the step limit; the translation may be incomplete.")

Step 1: ▁मैं
Text so far: मैं

Step 2: ▁अपने
Text so far: मैं अपने

Step 3: ▁खाते
Text so far: मैं अपने खाते

Step 4: ▁में
Text so far: मैं अपने खाते में

Step 5: ▁लॉ
Text so far: मैं अपने खाते में लॉ

Step 6: ग
Text so far: मैं अपने खाते में लॉग

Step 7: ▁इन
Text so far: मैं अपने खाते में लॉग इन

Step 8: ▁नहीं
Text so far: मैं अपने खाते में लॉग इन नहीं

Step 9: ▁कर
Text so far: मैं अपने खाते में लॉग इन नहीं कर

Step 10: ▁सकते
Text so far: मैं अपने खाते में लॉग इन नहीं कर सकते

Step 11: .
Text so far: मैं अपने खाते में लॉग इन नहीं कर सकते.

Step 12: ▁कृपया
Text so far: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया

Step 13: ▁मदद
Text so far: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद

Step 14: !
Text so far: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद!

Step 15: </s>
Text so far: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद!

Final translation: मैं अपने खाते में लॉग इन नहीं कर सकते. कृपया मदद!
Stopped because the model generated its end token.


In [20]:
# Enable attention-weight inspection
model.set_attn_implementation("eager")
model.eval()

# Reuse the first three IDs from the previous decoding exercise
# Expected prefix: </s> hin_Deva ▁मैं
decoder_prefix = decoder_ids[:, :3]

source_tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].tolist()
)

prefix_tokens = tokenizer.convert_ids_to_tokens(
    decoder_prefix[0].tolist()
)

print("Decoder prefix:", prefix_tokens)

# Reuse the saved encoder output and inspect decoder attention
with torch.inference_mode():
    outputs = model(
        encoder_outputs=source_outputs,
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=decoder_prefix,
        output_attentions=True,
        use_cache=False,
        return_dict=True
    )

# Retrieve cross-attention from the first decoder layer
first_layer_cross_attention = outputs.cross_attentions[0]

print("Decoder layers:", len(outputs.cross_attentions))
print("Cross-attention shape:", first_layer_cross_attention.shape)

# Select the first sentence, first head, and last decoder position
weights = first_layer_cross_attention[0, 0, -1, :].float().cpu()

print("\nQuery position contains:", prefix_tokens[-1])
print("Layer index: 0")
print("Head index: 0")
print("\nCross-attention to English source positions:")

for token, weight in zip(source_tokens, weights.tolist()):
    print(f"{token:15s} {weight:.4f}")

print("\nSum of weights:", weights.sum().item())

# Inspect the next token predicted from this decoder prefix
next_token_id = outputs.logits[0, -1, :].argmax().item()

print(
    "\nPredicted next token:",
    tokenizer.convert_ids_to_tokens(next_token_id)
)

Decoder prefix: ['</s>', 'hin_Deva', '▁मैं']
Decoder layers: 12
Cross-attention shape: torch.Size([1, 16, 3, 12])

Query position contains: ▁मैं
Layer index: 0
Head index: 0

Cross-attention to English source positions:
eng_Latn        0.0176
▁I              0.1038
▁cannot         0.0299
▁log            0.0006
▁into           0.0037
▁my             0.1278
▁account        0.0005
.               0.0080
▁Please         0.0060
▁help           0.0017
!               0.0140
</s>            0.6860

Sum of weights: 0.9997754096984863

Predicted next token: ▁अपने


In [21]:
# Reuse outputs from the previous cross-attention inspection
# Select decoder self-attention from the first layer
self_attention = outputs.decoder_attentions[0]

print("Decoder self-attention shape:", self_attention.shape)

# Select the first sentence and first attention head
attention_matrix = self_attention[0, 0].float().cpu()

# Retrieve labels for the decoder prefix
prefix_tokens = tokenizer.convert_ids_to_tokens(
    decoder_prefix[0].tolist()
)

print("\nDecoder tokens:", prefix_tokens)
print("Layer index: 0")
print("Head index: 0")

# Print the attention matrix with token labels
print("\nRows = query positions; columns = attended positions")
print(" " * 16 + "".join(f"{token:>12}" for token in prefix_tokens))

for token, row in zip(prefix_tokens, attention_matrix.tolist()):
    formatted_weights = "".join(f"{weight:12.4f}" for weight in row)
    print(f"{token:16}{formatted_weights}")

# Each row should sum to approximately one
print("\nRow sums:", attention_matrix.sum(dim=-1).tolist())

# Inspect weights assigned to future positions
future_weights = torch.triu(attention_matrix, diagonal=1)

print(
    "Largest absolute attention weight to a future position:",
    future_weights.abs().max().item()
)

Decoder self-attention shape: torch.Size([1, 16, 3, 3])

Decoder tokens: ['</s>', 'hin_Deva', '▁मैं']
Layer index: 0
Head index: 0

Rows = query positions; columns = attended positions
                        </s>    hin_Deva        ▁मैं
</s>                  1.0000      0.0000      0.0000
hin_Deva              0.0229      0.9771      0.0000
▁मैं                  0.6787      0.1274      0.1937

Row sums: [1.0, 0.999908447265625, 0.9998779296875]
Largest absolute attention weight to a future position: 0.0


In [22]:
import warnings

# Map readable language names to NLLB language codes
language_codes = {
    "english": "eng_Latn",
    "french": "fra_Latn",
    "spanish": "spa_Latn",
    "hindi": "hin_Deva",
    "tamil": "tam_Taml"
}

# Verify that every language code exists in the loaded tokenizer
for language, code in language_codes.items():
    code_id = tokenizer.convert_tokens_to_ids(code)

    if code_id == tokenizer.unk_token_id:
        raise ValueError(f"Unknown language code for {language}: {code}")

model.eval()


def translate_text(text, source_language, target_language):

    # Normalise the language names
    source_language = source_language.strip().lower()
    target_language = target_language.strip().lower()

    # Validate the message and selected languages
    if not text.strip():
        raise ValueError("Please provide a non-empty message.")

    if source_language not in language_codes:
        raise ValueError(f"Unsupported source language: {source_language}")

    if target_language not in language_codes:
        raise ValueError(f"Unsupported target language: {target_language}")

    if source_language == target_language:
        return text

    # Tokenise the message using its source language
    tokenizer.src_lang = language_codes[source_language]

    encoded = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Avoid silently truncating long messages
    if encoded["input_ids"].shape[1] > 512:
        raise ValueError("Please use a message containing at most 512 tokens.")

    # Retrieve the target-language marker
    target_id = tokenizer.convert_tokens_to_ids(
        language_codes[target_language]
    )

    # Generate the translation using greedy decoding
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            forced_bos_token_id=target_id,
            max_new_tokens=128,
            num_beams=1,
            do_sample=False
        )

    output_ids = generated[0].tolist()

    # Report issues that readable decoding could hide
    if tokenizer.unk_token_id in output_ids:
        warnings.warn("The generated output contains an unknown token.")

    if output_ids[-1] != model.config.eos_token_id:
        warnings.warn("Generation reached its limit; output may be incomplete.")

    return tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    )


# Four language pairs tested in both directions
test_cases = [
    ("english", "french", "Please help me reset my password."),
    ("french", "english", "Veuillez m'aider à réinitialiser mon mot de passe."),

    ("english", "spanish", "Please help me reset my password."),
    ("spanish", "english", "Por favor, ayúdeme a restablecer mi contraseña."),

    ("english", "hindi", "Please help me reset my password."),
    ("hindi", "english", "कृपया मेरा पासवर्ड रीसेट करने में मेरी मदद करें।"),

    ("english", "tamil", "Please help me reset my password."),
    ("tamil", "english", "எனது கடவுச்சொல்லை மீட்டமைக்க உதவுங்கள்.")
]

# Store results so we can review them later
baseline_results = []

for source, target, message in test_cases:

    translation = translate_text(message, source, target)

    baseline_results.append({
        "source_language": source,
        "target_language": target,
        "input": message,
        "translation": translation
    })

    print(f"\n{source.title()} → {target.title()}")
    print("Input:", message)
    print("Translation:", translation)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



English → French
Input: Please help me reset my password.
Translation: S'il vous plaît, aidez-moi à réinitialiser mon mot de passe.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



French → English
Input: Veuillez m'aider à réinitialiser mon mot de passe.
Translation: Please help me reset my password.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


English → Hindi
Input: Please help me reset my password.
Translation: कृपया मुझे अपना पासवर्ड रीसेट करने में मदद करें।


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Hindi → English
Input: कृपया मेरा पासवर्ड रीसेट करने में मेरी मदद करें।
Translation: Please help me reset my password.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



English → Tamil
Input: Please help me reset my password.
Translation: தயவு செய்து என் கடவுச்சொல்லை மீட்டமைக்க எனக்கு உதவுங்கள்.

Tamil → English
Input: எனது கடவுச்சொல்லை மீட்டமைக்க உதவுங்கள்.
Translation: Please help me reset my password.


we’ll now detect the source language automatically and pass it to our existing translation function.

In [23]:
%pip install -q langdetect==1.0.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [24]:

from langdetect import detect_langs, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Make repeated detection reproducible
DetectorFactory.seed = 0

# Map detection codes to names accepted by translate_text()
detected_language_names = {
    "en": "english",
    "fr": "french",
    "es": "spanish",
    "hi": "hindi",
    "ta": "tamil"
}


def detect_source_language(text, minimum_score=0.90):

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please provide a non-empty text message.")

    # Obtain ranked language candidates
    try:
        candidates = detect_langs(text)
    except LangDetectException as error:
        raise ValueError(
            "Could not detect a language from this message."
        ) from error

    best = candidates[0]

    # Reject languages outside our supported set
    if best.lang not in detected_language_names:
        raise ValueError(
            f"Detector suggested unsupported language code: {best.lang}"
        )

    # Provisional uncertainty check
    if best.prob < minimum_score:
        raise ValueError(
            f"Detection is uncertain: {best.lang}, score {best.prob:.3f}"
        )

    source_language = detected_language_names[best.lang]

    return source_language, best.prob


def auto_translate(text, target_language):

    # Detect the source language
    source_language, score = detect_source_language(text)

    # Reuse our existing translation function
    translation = translate_text(
        text,
        source_language,
        target_language
    )

    return {
        "detected_language": source_language,
        "detection_score": score,
        "translation": translation
    }


# Test each supported source language
# Expected source is recorded for checking, not passed to the detector
detection_tests = [
    (
        "english",
        "Please help me reset my password.",
        "french"
    ),
    (
        "french",
        "Veuillez m'aider à réinitialiser mon mot de passe.",
        "english"
    ),
    (
        "spanish",
        "Por favor, ayúdeme a restablecer mi contraseña.",
        "english"
    ),
    (
        "hindi",
        "कृपया मेरा पासवर्ड रीसेट करने में मेरी मदद करें।",
        "english"
    ),
    (
        "tamil",
        "எனது கடவுச்சொல்லை மீட்டமைக்க உதவுங்கள்.",
        "english"
    )
]

auto_results = []

for expected_source, message, target in detection_tests:

    print("\nInput:", message)
    print("Expected source:", expected_source)
    print("Target:", target)

    try:
        result = auto_translate(message, target)

        auto_results.append({
            "input": message,
            "expected_source": expected_source,
            "target_language": target,
            **result
        })

        print("Detected source:", result["detected_language"])
        print(
            "Detection matches expected:",
            result["detected_language"] == expected_source
        )
        print("Detection score:", round(result["detection_score"], 3))
        print("Translation:", result["translation"])

    except ValueError as error:
        print("Could not translate:", error)


Input: Please help me reset my password.
Expected source: english
Target: french


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: english
Detection matches expected: True
Detection score: 1.0
Translation: S'il vous plaît, aidez-moi à réinitialiser mon mot de passe.

Input: Veuillez m'aider à réinitialiser mon mot de passe.
Expected source: french
Target: english


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: french
Detection matches expected: True
Detection score: 1.0
Translation: Please help me reset my password.

Input: Por favor, ayúdeme a restablecer mi contraseña.
Expected source: spanish
Target: english


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: spanish
Detection matches expected: True
Detection score: 1.0
Translation: Please help me reset my password.

Input: कृपया मेरा पासवर्ड रीसेट करने में मेरी मदद करें।
Expected source: hindi
Target: english


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: hindi
Detection matches expected: True
Detection score: 1.0
Translation: Please help me reset my password.

Input: எனது கடவுச்சொல்லை மீட்டமைக்க உதவுங்கள்.
Expected source: tamil
Target: english
Detected source: tamil
Detection matches expected: True
Detection score: 1.0
Translation: Please help me reset my password.


In [25]:
# Test the current pipeline before adding term protection
message = "My FlyRank account shows error E403. Please help!"
protected_terms = ["FlyRank", "E403"]

target_languages = ["french", "spanish", "hindi", "tamil"]

term_test_results = []

for target in target_languages:

    print(f"\nEnglish → {target.title()}")
    print("Input:", message)

    try:
        # Detect the source language and translate
        result = auto_translate(message, target)
        translation = result["translation"]

        print("Detected source:", result["detected_language"])
        print("Translation:", translation)

        # Check whether each required term appears exactly
        term_checks = {}

        for term in protected_terms:
            preserved = term in translation
            term_checks[term] = preserved

            print(f"Exact text '{term}' preserved:", preserved)

        all_preserved = all(term_checks.values())
        print("All required terms preserved:", all_preserved)

        # Save the output for later comparison
        term_test_results.append({
            "target_language": target,
            "input": message,
            "translation": translation,
            "term_checks": term_checks,
            "all_preserved": all_preserved
        })

    except ValueError as error:
        print("Could not translate:", error)

        term_test_results.append({
            "target_language": target,
            "input": message,
            "error": str(error)
        })

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



English → French
Input: My FlyRank account shows error E403. Please help!


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: english
Translation: Mon compte FlyRank montre une erreur E403.
Exact text 'FlyRank' preserved: True
Exact text 'E403' preserved: True
All required terms preserved: True

English → Spanish
Input: My FlyRank account shows error E403. Please help!


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: english
Translation: Mi cuenta de FlyRank muestra el error E403. ¡Por favor, ayúdame!
Exact text 'FlyRank' preserved: True
Exact text 'E403' preserved: True
All required terms preserved: True

English → Hindi
Input: My FlyRank account shows error E403. Please help!


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: english
Translation: मेरे फ्लाई रैंक खाते में त्रुटि E403 दिखाई देती है। कृपया मदद करें!
Exact text 'FlyRank' preserved: False
Exact text 'E403' preserved: True
All required terms preserved: False

English → Tamil
Input: My FlyRank account shows error E403. Please help!
Detected source: english
Translation: என் FlyRank கணக்கில் பிழை E403 காட்டுகிறது. தயவு செய்து உதவி!
Exact text 'FlyRank' preserved: True
Exact text 'E403' preserved: True
All required terms preserved: True


In [26]:
#Placeholder based term protection
def translate_with_protection(text, target_language, terms):

    # Detect language from the original message
    source_language, score = detect_source_language(text)

    # Reserve this marker prefix to avoid collisions with user text
    if "ZXQTERM" in text:
        raise ValueError("Input already contains our reserved marker prefix.")

    # Remove duplicate terms and put longer terms first
    unique_terms = sorted(set(terms), key=len, reverse=True)

    if any(not term.strip() for term in unique_terms):
        raise ValueError("Protected terms must not be empty.")

    masked_text = text
    replacements = {}

    # Replace each term with an exact temporary marker
    for index, term in enumerate(unique_terms):
        if term in masked_text:
            marker = f"ZXQTERM{index}QXZ"
            masked_text = masked_text.replace(term, marker)
            replacements[marker] = term

    # Translate the message containing markers
    translated = translate_text(
        masked_text,
        source_language,
        target_language
    )

    # Check every marker before restoring any terms
    for marker in replacements:
        expected_count = masked_text.count(marker)
        actual_count = translated.count(marker)

        if actual_count != expected_count:
            raise ValueError(
                f"Marker validation failed for {marker}: "
                f"expected {expected_count}, found {actual_count}. "
                f"Raw translation: {translated}"
            )

    # Restore original spelling after validation
    restored = translated

    for marker, original_term in replacements.items():
        restored = restored.replace(marker, original_term)

    # Catch any remaining marker fragments using the reserved prefix
    if "ZXQTERM" in restored:
        raise ValueError(
            f"Unexpected marker text remains. Raw translation: {translated}"
        )

    return {
        "detected_language": source_language,
        "masked_input": masked_text,
        "raw_translation": translated,
        "translation": restored
    }


# Use the same example as our unprotected test
message = "My FlyRank account shows error E403. Please help!"
protected_terms = ["FlyRank", "E403"]
target_languages = ["french", "spanish", "hindi", "tamil"]

# Retrieve previous translations for comparison
previous_translations = {
    result["target_language"]: result["translation"]
    for result in term_test_results
    if "translation" in result
}

protected_results = []

for target in target_languages:

    print(f"\nEnglish → {target.title()}")
    print("Original:", message)
    print("Previous translation:", previous_translations.get(target))

    try:
        result = translate_with_protection(
            message,
            target,
            protected_terms
        )

        print("Masked input:", result["masked_input"])
        print("Raw translation:", result["raw_translation"])
        print("Restored translation:", result["translation"])

        for term in protected_terms:
            print(
                f"Exact text '{term}' present:",
                term in result["translation"]
            )

        protected_results.append({
            "target_language": target,
            "marker_validation_passed": True,
            **result
        })

    except ValueError as error:
        print("Protection failed:", error)

        protected_results.append({
            "target_language": target,
            "marker_validation_passed": False,
            "error": str(error)
        })

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



English → French
Original: My FlyRank account shows error E403. Please help!
Previous translation: Mon compte FlyRank montre une erreur E403.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Masked input: My ZXQTERM0QXZ account shows error ZXQTERM1QXZ. Please help!
Raw translation: Mon compte ZXQTERM0QXZ montre une erreur ZXQTERM1QXZ. Aidez-moi s'il vous plaît!
Restored translation: Mon compte FlyRank montre une erreur E403. Aidez-moi s'il vous plaît!
Exact text 'FlyRank' present: True
Exact text 'E403' present: True

English → Spanish
Original: My FlyRank account shows error E403. Please help!
Previous translation: Mi cuenta de FlyRank muestra el error E403. ¡Por favor, ayúdame!


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Masked input: My ZXQTERM0QXZ account shows error ZXQTERM1QXZ. Please help!
Raw translation: Mi cuenta ZXQTERM0QXZ muestra error ZXQTERM1QXZ. ¡Por favor ayúdame!
Restored translation: Mi cuenta FlyRank muestra error E403. ¡Por favor ayúdame!
Exact text 'FlyRank' present: True
Exact text 'E403' present: True

English → Hindi
Original: My FlyRank account shows error E403. Please help!
Previous translation: मेरे फ्लाई रैंक खाते में त्रुटि E403 दिखाई देती है। कृपया मदद करें!


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Masked input: My ZXQTERM0QXZ account shows error ZXQTERM1QXZ. Please help!
Raw translation: मेरे ZXQTERM0QXZ खाते में त्रुटि ZXQTERM1QXZ दिखाता है. कृपया मदद करें!
Restored translation: मेरे FlyRank खाते में त्रुटि E403 दिखाता है. कृपया मदद करें!
Exact text 'FlyRank' present: True
Exact text 'E403' present: True

English → Tamil
Original: My FlyRank account shows error E403. Please help!
Previous translation: என் FlyRank கணக்கில் பிழை E403 காட்டுகிறது. தயவு செய்து உதவி!
Masked input: My ZXQTERM0QXZ account shows error ZXQTERM1QXZ. Please help!
Raw translation: என் ZXQTERM0QXZ கணக்கில் பிழை ZXQTERM1QXZ காட்டுகிறது. தயவு செய்து உதவி!
Restored translation: என் FlyRank கணக்கில் பிழை E403 காட்டுகிறது. தயவு செய்து உதவி!
Exact text 'FlyRank' present: True
Exact text 'E403' present: True


In [27]:
# Difficult examples for our current translation pipeline
robustness_cases = [
    {
        "case": "Informal English",
        "text": "I cant log in to FlyRank, pls help! It shows E403 😢",
        "target": "hindi",
        "terms": ["FlyRank", "E403"]
    },
    {
        "case": "Very short message",
        "text": "Help!",
        "target": "hindi",
        "terms": []
    },
    {
        "case": "Mixed English and Hindi",
        "text": "मेरा FlyRank account login नहीं हो रहा। Please help!",
        "target": "english",
        "terms": ["FlyRank"]
    },
    {
        "case": "Romanised Hindi",
        "text": "Mera account nahi khul raha hai, kripya madad karein.",
        "target": "english",
        "terms": []
    },
    {
        "case": "Repeated terms",
        "text": "FlyRank shows E403 again. FlyRank still shows E403.",
        "target": "hindi",
        "terms": ["FlyRank", "E403"]
    },
    {
        "case": "Urgent message",
        "text": (
            "This is urgent! My FlyRank account is locked, "
            "and I cannot access my work. Please help immediately!"
        ),
        "target": "french",
        "terms": ["FlyRank"]
    },
    {
        "case": "Error code only",
        "text": "E403",
        "target": "tamil",
        "terms": ["E403"]
    }
]

robustness_results = []

for case in robustness_cases:

    print("\nCase:", case["case"])
    print("Input:", case["text"])
    print("Target:", case["target"])

    try:
        result = translate_with_protection(
            case["text"],
            case["target"],
            case["terms"]
        )

        translation = result["translation"]

        print("Detected source:", result["detected_language"])
        print("Translation:", translation)

        # Check exact occurrence counts for each protected term
        term_counts = {}

        for term in case["terms"]:
            expected_count = case["text"].count(term)
            actual_count = translation.count(term)

            term_counts[term] = {
                "expected": expected_count,
                "actual": actual_count,
                "matches": expected_count == actual_count
            }

            print(
                f"Term '{term}': expected {expected_count}, "
                f"found {actual_count}"
            )

        print("Execution status: Completed")
        print("Translation quality: Pending review")

        robustness_results.append({
            **case,
            "status": "completed",
            "detected_language": result["detected_language"],
            "translation": translation,
            "term_counts": term_counts,
            "quality_review": "pending"
        })

    except ValueError as error:
        print("Execution status: Rejected")
        print("Reason:", error)

        robustness_results.append({
            **case,
            "status": "rejected",
            "reason": str(error)
        })

completed = sum(
    result["status"] == "completed"
    for result in robustness_results
)

print("\nTotal cases:", len(robustness_results))
print("Completed:", completed)
print("Rejected:", len(robustness_results) - completed)
print("These counts measure execution outcomes, not translation accuracy.")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Case: Informal English
Input: I cant log in to FlyRank, pls help! It shows E403 😢
Target: hindi


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: english
Translation: मैं FlyRank में लॉग इन नहीं कर सकते, कृपया मदद! यह E403 दिखाता है 😢
Term 'FlyRank': expected 1, found 1
Term 'E403': expected 1, found 1
Execution status: Completed
Translation quality: Pending review

Case: Very short message
Input: Help!
Target: hindi
Execution status: Rejected
Reason: Detector suggested unsupported language code: nl

Case: Mixed English and Hindi
Input: मेरा FlyRank account login नहीं हो रहा। Please help!
Target: english
Detected source: english
Translation: मेरा FlyRank account login नहीं हो रहा। Please help!
Term 'FlyRank': expected 1, found 1
Execution status: Completed
Translation quality: Pending review

Case: Romanised Hindi
Input: Mera account nahi khul raha hai, kripya madad karein.
Target: english
Execution status: Rejected
Reason: Detector suggested unsupported language code: id

Case: Repeated terms
Input: FlyRank shows E403 again. FlyRank still shows E403.
Target: hindi


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected source: english
Translation: FlyRank फिर से E403 दिखाता है। FlyRank अभी भी E403 दिखाता है।
Term 'FlyRank': expected 2, found 2
Term 'E403': expected 2, found 2
Execution status: Completed
Translation quality: Pending review

Case: Urgent message
Input: This is urgent! My FlyRank account is locked, and I cannot access my work. Please help immediately!
Target: french
Detected source: english
Translation: Mon compte FlyRank est verrouillé et je ne peux pas accéder à mon travail.
Term 'FlyRank': expected 1, found 1
Execution status: Completed
Translation quality: Pending review

Case: Error code only
Input: E403
Target: tamil
Execution status: Rejected
Reason: Detection is uncertain: es, score 0.867

Total cases: 7
Completed: 4
Rejected: 3
These counts measure execution outcomes, not translation accuracy.


In [28]:
import re


def translate_checked(text, target_language, protected_terms=None):

    protected_terms = list(dict.fromkeys(protected_terms or []))

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please provide a non-empty text message.")

    target_language = target_language.strip().lower()

    if target_language not in language_codes:
        raise ValueError("Unsupported target language.")

    if any(not term.strip() for term in protected_terms):
        raise ValueError("Protected terms must not be empty.")

    # 1. A message containing only a protected term needs no translation
    if text.strip() in protected_terms:
        return {
            "status": "preserved_only",
            "source_language": "not needed",
            "translation": text,
            "sentence_count": 0
        }

    # Reserve our placeholder prefix
    if "ZXQTERM" in text:
        raise ValueError("Input contains our reserved marker prefix.")

    # Remove protected terms only from a copy used for script checks
    language_text = text

    for term in sorted(protected_terms, key=len, reverse=True):
        language_text = language_text.replace(term, " ")

    script_patterns = {
        "Latin": r"[A-Za-z]",
        "Devanagari": r"[\u0900-\u097F]",
        "Tamil": r"[\u0B80-\u0BFF]"
    }

    found_scripts = [
        name
        for name, pattern in script_patterns.items()
        if re.search(pattern, language_text)
    ]

    # 2. Block the observed mixed-script failure
    if len(found_scripts) > 1:
        raise ValueError(
            "Mixed scripts detected: "
            + ", ".join(found_scripts)
            + ". This version cannot reliably translate this mixture."
        )

    # Detect language once from the full original message
    source_language, score = detect_source_language(text)

    # Insert protected-term markers
    masked_text = text
    replacements = {}

    for index, term in enumerate(
        sorted(protected_terms, key=len, reverse=True)
    ):
        if term in masked_text:
            marker = f"ZXQTERM{index}QXZ"
            masked_text = masked_text.replace(term, marker)
            replacements[marker] = term

    # 3. Split the masked text into sentences
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?।])\s+", masked_text)
        if sentence.strip()
    ]

    translated_sentences = []

    for sentence in sentences:
        translated_sentence = translate_text(
            sentence,
            source_language,
            target_language
        )

        # Validate markers separately for each sentence
        for marker in replacements:
            if translated_sentence.count(marker) != sentence.count(marker):
                raise ValueError(
                    f"Marker validation failed for {marker}. "
                    f"Raw sentence: {translated_sentence}"
                )

        translated_sentences.append(translated_sentence)

    translation = " ".join(translated_sentences)

    # Restore the original terms
    for marker, term in replacements.items():
        translation = translation.replace(marker, term)

    if "ZXQTERM" in translation:
        raise ValueError("Unexpected marker text remains.")

    return {
        "status": "completed",
        "source_language": source_language,
        "translation": translation,
        "sentence_count": len(sentences)
    }


# Recheck the specific failures
fix_tests = [
    {
        "case": "Protected code only",
        "text": "E403",
        "target": "tamil",
        "terms": ["E403"]
    },
    {
        "case": "Mixed-script message",
        "text": "मेरा FlyRank account login नहीं हो रहा। Please help!",
        "target": "english",
        "terms": ["FlyRank"]
    },
    {
        "case": "Urgent message",
        "text": (
            "This is urgent! My FlyRank account is locked, "
            "and I cannot access my work. Please help immediately!"
        ),
        "target": "french",
        "terms": ["FlyRank"]
    },
    {
        "case": "Repeated terms",
        "text": "FlyRank shows E403 again. FlyRank still shows E403.",
        "target": "hindi",
        "terms": ["FlyRank", "E403"]
    }
]

# Retrieve earlier outputs for comparison
previous_results = {
    result["case"]: result.get("translation", result.get("reason"))
    for result in robustness_results
}

checked_results = []

for case in fix_tests:

    print("\nCase:", case["case"])
    print("Input:", case["text"])

    if case["case"] in previous_results:
        print("Previous result:", previous_results[case["case"]])

    try:
        result = translate_checked(
            case["text"],
            case["target"],
            case["terms"]
        )

        print("Status:", result["status"])
        print("Source language:", result["source_language"])
        print("Sentences processed:", result["sentence_count"])
        print("New output:", result["translation"])

        checked_results.append({**case, **result})

    except ValueError as error:
        print("Status: Rejected")
        print("Reason:", error)

        checked_results.append({
            **case,
            "status": "rejected",
            "reason": str(error)
        })


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Case: Protected code only
Input: E403
Status: preserved_only
Source language: not needed
Sentences processed: 0
New output: E403

Case: Mixed-script message
Input: मेरा FlyRank account login नहीं हो रहा। Please help!
Status: Rejected
Reason: Mixed scripts detected: Latin, Devanagari. This version cannot reliably translate this mixture.

Case: Urgent message
Input: This is urgent! My FlyRank account is locked, and I cannot access my work. Please help immediately!
Previous result: Mon compte FlyRank est verrouillé et je ne peux pas accéder à mon travail.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Status: completed
Source language: english
Sentences processed: 3
New output: C'est urgent ! Mon compte FlyRank est verrouillé, et je ne peux pas accéder à mon travail. Aidez-moi immédiatement !

Case: Repeated terms
Input: FlyRank shows E403 again. FlyRank still shows E403.
Previous result: FlyRank फिर से E403 दिखाता है। FlyRank अभी भी E403 दिखाता है।


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Status: completed
Source language: english
Sentences processed: 2
New output: FlyRank फिर से E403 दिखाता है। FlyRank अभी भी E403 दिखाता है।


In [29]:
#Evaluation
%pip install -q sacrebleu

from sacrebleu.metrics import BLEU, CHRF

# Assistant-drafted references for a small development exercise.
# These examples are not an independent final test set.
evaluation_examples = [
    {
        "source": "Please help me reset my password.",
        "reference": (
            "Veuillez m'aider à réinitialiser mon mot de passe."
        ),
        "terms": []
    },
    {
        "source": (
            "My FlyRank account shows error E403. Please help!"
        ),
        "reference": (
            "Mon compte FlyRank affiche l'erreur E403. "
            "Aidez-moi, s'il vous plaît !"
        ),
        "terms": ["FlyRank", "E403"]
    },
    {
        "source": (
            "This is urgent! My FlyRank account is locked, "
            "and I cannot access my work. Please help immediately!"
        ),
        "reference": (
            "C'est urgent ! Mon compte FlyRank est verrouillé "
            "et je ne peux pas accéder à mon travail. "
            "Veuillez m'aider immédiatement !"
        ),
        "terms": ["FlyRank"]
    }
]

baseline_predictions = []
updated_predictions = []
references = []
evaluation_records = []

for index, example in enumerate(evaluation_examples, start=1):

    source = example["source"]
    reference = example["reference"]

    # Original translation wrapper with the known source language
    baseline = translate_text(
        source,
        "english",
        "french"
    )

    # Updated pipeline: detection, protection and sentence splitting
    updated_result = translate_checked(
        source,
        "french",
        example["terms"]
    )

    updated = updated_result["translation"]

    baseline_predictions.append(baseline)
    updated_predictions.append(updated)
    references.append(reference)

    evaluation_records.append({
        "source": source,
        "reference": reference,
        "baseline": baseline,
        "updated": updated
    })

    print(f"\nExample {index}")
    print("Source:", source)
    print("Reference:", reference)
    print("Baseline:", baseline)
    print("Updated:", updated)

# Specify metric settings explicitly
bleu = BLEU(tokenize="13a")
chrf = CHRF(char_order=6, word_order=0, beta=2)

metric_results = {}

for name, predictions in [
    ("Original wrapper", baseline_predictions),
    ("Updated pipeline", updated_predictions)
]:
    bleu_score = bleu.corpus_score(predictions, [references]).score
    chrf_score = chrf.corpus_score(predictions, [references]).score

    metric_results[name] = {
        "BLEU": bleu_score,
        "chrF": chrf_score
    }

    print(f"\n{name}")
    print(f"BLEU: {bleu_score:.2f}")
    print(f"chrF: {chrf_score:.2f}")

print("\nBLEU settings:", bleu.get_signature())
print("chrF settings:", chrf.get_signature())

print("\nExamples evaluated:", len(references))
print("Direction: English → French")
print("Development exercise only — not final test accuracy.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.1 MB/s eta 0:00:00


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 1
Source: Please help me reset my password.
Reference: Veuillez m'aider à réinitialiser mon mot de passe.
Baseline: S'il vous plaît, aidez-moi à réinitialiser mon mot de passe.
Updated: S'il vous plaît, aidez-moi à réinitialiser mon mot de passe.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 2
Source: My FlyRank account shows error E403. Please help!
Reference: Mon compte FlyRank affiche l'erreur E403. Aidez-moi, s'il vous plaît !
Baseline: Mon compte FlyRank montre une erreur E403.
Updated: Mon compte FlyRank montre une erreur E403. S'il vous plaît, aidez-moi !


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 3
Source: This is urgent! My FlyRank account is locked, and I cannot access my work. Please help immediately!
Reference: C'est urgent ! Mon compte FlyRank est verrouillé et je ne peux pas accéder à mon travail. Veuillez m'aider immédiatement !
Baseline: Mon compte FlyRank est verrouillé et je ne peux pas accéder à mon travail.
Updated: C'est urgent ! Mon compte FlyRank est verrouillé, et je ne peux pas accéder à mon travail. Aidez-moi immédiatement !

Original wrapper
BLEU: 53.17
chrF: 59.16

Updated pipeline
BLEU: 54.45
chrF: 74.25

BLEU settings: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
chrF settings: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0

Examples evaluated: 3
Direction: English → French
Development exercise only — not final test accuracy.
